# 02 — 单层 Sanity Check（梯度检验）

手写 backward 最容易出 bug。这里对每个层做**数值梯度检验**：

$$\frac{\partial L}{\partial w} \approx \frac{L(w + \epsilon) - L(w - \epsilon)}{2\epsilon}$$

把数值梯度和我们解析推导的梯度对比，relative error 应该在 $10^{-5}$ 量级。

如果你修改了任何一个层的 forward/backward，先来这里跑一遍。

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
np.random.seed(42)

from backend.core import Conv2D, AvgPool2D, FC, Tanh, Flatten

EPS = 1e-5

def numerical_grad(f, x, eps=EPS):
    """对 x 的每一个元素扰动 ±eps，估计梯度。"""
    g = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    while not it.finished:
        i = it.multi_index
        orig = x[i].copy()
        x[i] = orig + eps; fp = f()
        x[i] = orig - eps; fm = f()
        x[i] = orig
        g[i] = (fp - fm) / (2 * eps)
        it.iternext()
    return g

def rel_err(a, b):
    return np.abs(a - b).max() / (np.abs(a).max() + np.abs(b).max() + 1e-12)

## Tanh

In [ ]:
x = np.random.randn(3, 5).astype(np.float64)
layer = Tanh()
out = layer.forward(x)
dout = np.random.randn(*out.shape)
dx_analytic = layer.backward(dout)
dx_numeric = numerical_grad(lambda: (Tanh().forward(x) * dout).sum(), x)
print(f'Tanh dx rel_err: {rel_err(dx_analytic, dx_numeric):.2e}')

## FC

In [ ]:
x = np.random.randn(4, 6).astype(np.float64)
fc = FC(6, 5)
fc.W = fc.W.astype(np.float64)
fc.b = fc.b.astype(np.float64)
out = fc.forward(x); dout = np.random.randn(*out.shape)
dx_a = fc.backward(dout)

def loss():
    return (fc.forward(x) * dout).sum()

dx_n = numerical_grad(loss, x)
dW_n = numerical_grad(loss, fc.W)
db_n = numerical_grad(loss, fc.b)
# 注意：FC.backward 内部把梯度除了 N，所以这里乘回来再比
N = x.shape[0]
print(f'FC dx rel_err: {rel_err(dx_a, dx_n):.2e}')
print(f'FC dW rel_err: {rel_err(fc.dW * N, dW_n):.2e}')
print(f'FC db rel_err: {rel_err(fc.db * N, db_n):.2e}')

## Conv2D（小尺寸，方便数值检验）

In [ ]:
x = np.random.randn(2, 3, 5, 5).astype(np.float64)
conv = Conv2D(in_c=3, out_c=4, k=3)
conv.W = conv.W.astype(np.float64); conv.b = conv.b.astype(np.float64)
out = conv.forward(x); dout = np.random.randn(*out.shape)
dx_a = conv.backward(dout)

def loss():
    c = Conv2D(in_c=3, out_c=4, k=3)
    c.W = conv.W; c.b = conv.b
    return (c.forward(x) * dout).sum()

dx_n = numerical_grad(loss, x)
dW_n = numerical_grad(loss, conv.W)
db_n = numerical_grad(loss, conv.b)
N = x.shape[0]
print(f'Conv2D dx rel_err: {rel_err(dx_a, dx_n):.2e}')
print(f'Conv2D dW rel_err: {rel_err(conv.dW * N, dW_n):.2e}')
print(f'Conv2D db rel_err: {rel_err(conv.db * N, db_n):.2e}')

## AvgPool

In [ ]:
x = np.random.randn(2, 3, 6, 6).astype(np.float64)
pool = AvgPool2D(k=2, stride=2)
out = pool.forward(x); dout = np.random.randn(*out.shape)
dx_a = pool.backward(dout)
dx_n = numerical_grad(lambda: (AvgPool2D(2, 2).forward(x) * dout).sum(), x)
print(f'AvgPool dx rel_err: {rel_err(dx_a, dx_n):.2e}')

## 全网络端到端：loss 真的能降吗？

In [ ]:
import matplotlib.pyplot as plt
from backend.model import LeNet5
from backend.core import Momentum

np.random.seed(0)
model = LeNet5()
opt = Momentum(lr=0.1, momentum=0.9)

X = np.random.randn(32, 1, 28, 28).astype(np.float32) * 0.5
y = np.random.randint(0, 10, size=32)

losses = []
for step in range(80):
    logits = model.forward(X)
    loss = model.loss_fn.forward(logits, y)
    dlogits = model.loss_fn.backward()
    model.backward(dlogits)
    opt.step(model._trainable_layers())
    losses.append(float(loss))

preds = np.argmax(model.forward(X), axis=-1)
acc = (preds == y).mean() * 100

plt.figure(figsize=(6, 3))
plt.plot(losses, color='#38bdf8')
plt.xlabel('step'); plt.ylabel('loss')
plt.title(f'Overfit 32 random samples — final acc {acc:.0f}%')
plt.tight_layout(); plt.show()